In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import re
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from bert_score import score
from torch import Tensor
import torch
import torch.nn.functional as F

In [2]:
def evaluate_predictions(y_true, y_pred):
    """
    Compute accuracy, micro/macro/weighted F1, precision, and recall scores
    for categorical answers (e.g., 'A', 'B', 'C', 'D').

    Args:
        y_true (list): True labels.
        y_pred (list): Predicted labels.

    Returns:
        dict: Dictionary containing all computed metrics.
    """
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_micro": f1_score(y_true, y_pred, average="micro"),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "precision_micro": precision_score(y_true, y_pred, average="micro"),
        "precision_macro": precision_score(y_true, y_pred, average="macro"),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted"),
        "recall_micro": recall_score(y_true, y_pred, average="micro"),
        "recall_macro": recall_score(y_true, y_pred, average="macro"),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted"),
    }
    return metrics

In [3]:
df = pd.read_csv("counselingBench.csv")
df_filled = df.fillna("")

In [4]:
correct_labels = df['Correct Answer (Letter)'].tolist()

In [ ]:
with open('output labels file') as f:
    predicted_labels = json.load(f)

In [6]:
results = evaluate_predictions(correct_labels, predicted_labels)

In [ ]:
for k, v in results.items():
    print(f"{k}: {v:.4f}")

## CoT reasoning chain evaluation (reference based)

In [ ]:
with open("output reasoning files", "r") as f:
    data = [json.loads(line) for line in f]

In [ ]:
data[0]

In [10]:
correct_reasonings = []
predicted_reasonings = []

In [11]:
for i in range(len(data)):
    correct_reasonings.append(re.sub(r"Therefore, the correct answer is \([A-D]\)\.?", "",data[i]["reference"]).strip())
    predicted_reasonings.append(data[i]["hypothesis"])

In [12]:
model = SentenceTransformer("all-mpnet-base-v2")

In [13]:
def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

In [ ]:
def generate_embedding(sentence, model):
    if isinstance(sentence, float):
        sentence = ""
    with torch.no_grad():
        embeddings = torch.Tensor(model.encode(sentence))
    return embeddings

In [15]:
def generate_embeddings_dict(true_list,pred_list, model):
    embedding_dict = dict([])
    for idx in range(len(true_list)):
        embedding_pred = generate_embedding(pred_list[idx], model)
        embedding_true = generate_embedding(true_list[idx], model)
        embedding_dict[idx] = (embedding_pred, embedding_true)
    return embedding_dict

In [16]:
def calculate_cosine_similarity(embedding_dict):
    similarity = []
    cos = torch.nn.CosineSimilarity(dim=0, eps=1e-6)
    for idx in range(len(embedding_dict)):
        similarity.append(cos(embedding_dict[idx][0], embedding_dict[idx][1]).item())
    return similarity

In [17]:
def calculate_bert_score(true_list,pred_list):
    bert_score = []
    for idx in range(len(true_list)):
        P, R, F1 = score([pred_list[idx]], [true_list[idx]], rescale_with_baseline=True, lang="en")
        bert_score.append(max(F1[0].item(), 0))
    return bert_score

In [18]:
def calculate_rouge_L_score(true_list,pred_list):
    rouge_score = []
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    for idx in range(len(true_list)):
        scores = scorer.score(pred_list[idx], true_list[idx])
        rouge_score.append(scores['rougeL'].fmeasure)
    return rouge_score

In [19]:
def calculate_rouge_1_score(true_list,pred_list):
    rouge_score = []
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    for idx in range(len(true_list)):
        scores = scorer.score(pred_list[idx], true_list[idx])
        rouge_score.append(scores['rouge1'].fmeasure)
    return rouge_score

In [20]:
def calculate_length_difference(true_list,pred_list):
    length_difference = []
    length_pred = []
    for idx in range(len(true_list)):
        e1 = len(pred_list[idx]) / 4
        e2 = len(true_list[idx]) / 4
        length_difference.append(e1 - e2)
        length_pred.append(e1)
    return length_difference, length_pred

In [21]:
embedding_dict = generate_embeddings_dict(correct_reasonings,predicted_reasonings, model)

In [22]:
cosine_sim = calculate_cosine_similarity(embedding_dict)

In [ ]:
np.mean(np.array(cosine_sim))

In [ ]:
bert_score = calculate_bert_score(correct_reasonings,predicted_reasonings)

In [ ]:
np.mean(np.array(bert_score))

In [26]:
rouge_L_score = calculate_rouge_L_score(correct_reasonings,predicted_reasonings)

In [ ]:
np.mean(np.array(rouge_L_score))

In [28]:
rouge_1_score = calculate_rouge_1_score(correct_reasonings,predicted_reasonings)

In [ ]:
np.mean(np.array(rouge_1_score))